<a href="https://colab.research.google.com/github/1heidi/inventory_2022/blob/inventory_update_2026/updating_inventory_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2026 GBC Inventory Update Pipeline

This notebook executes the automated text-mining and entity-extraction pipeline to update the GBC inventory. Because this repository relies on custom fine-tuned deep learning models built in 2022, new steps account for shifts in external packages, cloud environments, and the Hugging Face Hub.

* **Hugging Face Hub Protocol Shifts:** The legacy `transformers` library engine used in the original codebase cannot handle modern Hugging Face URL caching and security redirects, causing network download failures. Cell 4 bypasses this entirely by establishing a strict local staging workflow, ensuring the weights are pulled down safely and verified via local checksums.
* **Modern Package & Environment Conflicts:** Modern runtime environments use updated Python stacks (Python 3.10+) and strict PyTorch serialization rules. These modern updates block or crash when trying to read legacy 2022 model checkpoints that contain custom team tracking objects (like `inventory_utils.custom_classes.Metrics`).
* **Safeguards:** To fix this without rewriting the core scripts, this notebook downgrades the base engine, injects repository paths directly into Python's runtime memory (Cell 5), and strictly maps checkpoint pointers directly to binary files. This allows use of the original 2022 fine-tuned assets within a cloud container.

### Execution Protocol

1. (Recommended) Runtime -> Change runtime type -> GPU.
2. Run **Cell 1** to initialize the underlying Python 3.8 environment.
3. **Manually click "Restart session"** at the top of Google Colab when Cell 1 finishes. Confirmation must be provided in Cell 1B.
4. Run **Cells 2 through 6 in absolute sequential order.** Do not skip steps or re-run cells out of order, and be sure to adjust the yml and login in config files for the new date range.


In [ ]:
# ==============================================================================
# CELL 1: ENGINE INITIALIZATION & PYTHON DOWNGRADE
# ==============================================================================
# Why it's being done differently:
# Modern Google Colab runtimes use newer Python versions (like Python 3.10+)
# that are incompatible with the legacy 2022 dependency tree. We force-install
# a native Python 3.8 Linux Conda environment to preserve execution stability.

#### 🛑 CRITICAL ACTION: You must click the "Restart session" banner at the top
#of Colab immediately after this cell finishes to force the notebook interface
#to switch over to the newly installed Python engine.

# 1. Download and install a native Python 3.8 Conda environment
!wget -qO installer.sh https://repo.anaconda.com/miniconda/Miniconda3-py38_4.12.0-Linux-x86_64.sh
!bash installer.sh -b -f -p /usr/local

# 2. Configure conda and enforce Python 3.8 + pip alignment
!conda config --set always_yes yes
!conda install -y -c conda-forge python=3.8 pip

# 3. Print verified version (Must confirm: Python 3.8.x)
!python --version

PREFIX=/usr/local
Unpacking payload ...
Solving environment: / - \ | / - \ | done

## Package Plan ##

  environment location: /usr/local

  added / updated specs:
    - _libgcc_mutex==0.1=main
    - _openmp_mutex==4.5=1_gnu
    - brotlipy==0.7.0=py38h27cfd23_1003
    - ca-certificates==2022.3.29=h06a4308_1
    - certifi==2021.10.8=py38h06a4308_2
    - cffi==1.15.0=py38hd667e15_1
    - charset-normalizer==2.0.4=pyhd3eb1b0_0
    - colorama==0.4.4=pyhd3eb1b0_0
    - conda-content-trust==0.1.1=pyhd3eb1b0_0
    - conda-package-handling==1.8.1=py38h7f8727e_0
    - conda==4.12.0=py38h06a4308_0
    - cryptography==36.0.0=py38h9ce1e76_0
    - idna==3.3=pyhd3eb1b0_0
    - ld_impl_linux-64==2.35.1=h7274673_9
    - libffi==3.3=he6710b0_2
    - libgcc-ng==9.3.0=h5101ec6_17
    - libgomp==9.3.0=h5101ec6_17
    - libstdcxx-ng==9.3.0=hd4cf53a_17
    - ncurses==6.3=h7f8727e_2
    - openssl==1.1.1n=h7f8727e_0
    - pip==21.2.4=py38h06a4308_0
    - pycosat==0.6.3=py38h7b6447c_1
    - pyc

In [ ]:
# ==============================================================================
# CELL 1B: CONFIRM RESTART
# ==============================================================================

confirm = input(
    "Restart session completed? Type Y to continue: "
).strip()

if confirm != "Y":
    raise SystemExit(
        "STOPPED: You must type 'Y' after restarting the runtime."
    )

print("✔ Restart confirmed. Proceeding...")

Restart session completed? Type Y to continue: Y
✔ Restart confirmed. Proceeding...


In [ ]:
# ==============================================================================
# CELL 2: STORAGE MAPPING & DIRECTORY NAVIGATION
# ==============================================================================
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/GitHub/inventory_2022

Mounted at /content/drive
/content/drive/MyDrive/GitHub/inventory_2022


In [ ]:
# ==============================================================================
## CELL 2B: DOUBLE CHECK BRANCH
# ==============================================================================
%%bash
cd /content/drive/MyDrive/GitHub/inventory_2022

git branch --show-current

inventory_update_2026


In [ ]:
# ==============================================================================
# CELL 3: LEGACY LIBRARY PINNING
# ==============================================================================
# Why it's being done differently:
# Modern versions of Hugging Face 'transformers' have deprecated the original
# 2022 internal tokenizing properties. Pinning these exact legacy versions
# protects the model from crashing during inference text tokenization.

!pip install tokenizers==0.11.4 transformers==4.16.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 41.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 65.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 54.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 785.1/785.1 kB 42.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 13.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 39.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 806.0/806.0 kB 43.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 20.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 73.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 12.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
%%bash
# ==============================================================================
# CELL 4: PRODUCTION WEIGHTS REGISTRY & DEPENDENCY BOOTSTRAP
# ==============================================================================
# Why this cell exists:
#
# This pipeline depends on two distinct model systems:
#
# 1. Legacy PyTorch checkpoints (.pt files)
#    - Used directly by custom 2022 training/inference code
#    - Must be referenced via explicit file-path pointer files
#    - These pointer files are consumed by native Python open() calls
#
# 2. Hugging Face Transformer backbone models
#    - Required by AutoConfig / AutoModel loading in classification + NER steps
#    - Must exist as a fully materialized local directory OR valid HF repo access
#
# This cell ensures both systems are correctly initialized.
#
# Design constraints:
# - Avoid re-downloading existing large assets (efficiency + reproducibility)
# - Prevent pointer mismatches that break downstream Snakemake jobs
# - Ensure deterministic file paths for all model resolution steps
# - Guarantee compatibility between legacy PyTorch pipeline and HF Transformers
#
# Result:
# After this cell, all downstream pipeline steps (Snakemake rules) can safely:
# - Load .pt checkpoints via explicit path pointers
# - Load Transformer configs either locally or via resolved HF cache
# ==============================================================================
set -e  # fail fast if anything breaks

cd /content/drive/MyDrive/GitHub/inventory_2022 # Ensure we are in the correct directory

# ------------------------------------------------------------------------------
# 1. Repo setup (Directly execute commands from 'setup_for_updating' target)
# ------------------------------------------------------------------------------
python3.8 -m pip install -r requirements.txt
python3 -c "import nltk; nltk.download('punkt')"

# ------------------------------------------------------------------------------
# 2. CLASSIFIER WEIGHTS (PyTorch .pt)
# ------------------------------------------------------------------------------
mkdir -p out/classif_train_out/best

if [ ! -f out/classif_train_out/article_classifier.pt ]; then
    wget -O out/classif_train_out/article_classifier.pt \
    https://huggingface.co/globalbiodata/inventory/resolve/main/article_classifier.pt
fi

echo "5718a7f70becacb46d46501734c83aab81c86feec563594f6a25c116aa31b521 out/classif_train_out/article_classifier.pt" \
| sha256sum -c

echo "out/classif_train_out/article_classifier.pt" > out/classif_train_out/best/best_checkpt.txt

# ------------------------------------------------------------------------------
# 3. NER WEIGHTS (PyTorch .pt)
# ------------------------------------------------------------------------------
mkdir -p out/ner_train_out/best

if [ ! -f out/ner_train_out/named_entity_recognition.pt ]; then
    wget -O out/ner_train_out/named_entity_recognition.pt \
    https://huggingface.co/globalbiodata/inventory/resolve/main/named_entity_recognition.pt
fi

echo "dc0bc8b4929e33da52bc92e12720260b392421883889e0a36c809cb0b5c40f5d out/ner_train_out/named_entity_recognition.pt" \
| sha256sum -c

echo "out/ner_train_out/named_entity_recognition.pt" > out/ner_train_out/best/best_checkpt.txt

# ------------------------------------------------------------------------------
# 4. 🚨 CRITICAL FIX: HF TRANSFORMER BACKBONE (REQUIRED)
# ------------------------------------------------------------------------------
HF_MODEL_DIR="allenai/dsp_roberta_base_dapt_biomed_tapt_rct_500"

mkdir -p "$HF_MODEL_DIR"

# Download ONLY if missing (safe + idempotent)
if [ ! -f "$HF_MODEL_DIR/config.json" ]; then
    echo "Downloading Hugging Face backbone model..."

    wget -q -P "$HF_MODEL_DIR" \
    https://huggingface.co/allenai/dsp_roberta_base_dapt_biomed_tapt_rct_500/resolve/main/config.json

    wget -q -P "$HF_MODEL_DIR" \
    https://huggingface.co/allenai/dsp_roberta_base_dapt_biomed_tapt_rct_500/resolve/main/pytorch_model.bin

    wget -q -P "$HF_MODEL_DIR" \
    https://huggingface.co/allenai/dsp_roberta_base_dapt_biomed_tapt_rct_500/resolve/main/tokenizer_config.json

    wget -q -P "$HF_MODEL_DIR" \
    https://huggingface.co/allenai/dsp_roberta_base_dapt_biomed_tapt_rct_500/resolve/main/vocab.json

    wget -q -P "$HF_MODEL_DIR" \
    https://huggingface.co/allenai/dsp_roberta_base_dapt_biomed_tapt_rct_500/resolve/main/merges.txt

    wget -q -P "$HF_MODEL_DIR" \
    https://huggingface.co/allenai/dsp_roberta_base_dapt_biomed_tapt_rct_500/resolve/main/special_tokens_map.json
fi

# ------------------------------------------------------------------------------
# 5. Sanity check
# ------------------------------------------------------------------------------
test -f "$HF_MODEL_DIR/config.json" || {
    echo "❌ HF backbone missing"
    exit 1
}

echo "✔ CELL 4 COMPLETE: all weights + HF backbone ready"


out/classif_train_out/article_classifier.pt: OK
out/ner_train_out/named_entity_recognition.pt: OK
✔ CELL 4 COMPLETE: all weights + HF backbone ready


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [ ]:
# ==============================================================================
# CELL 5: RUNTIME ENVIRONMENT PATH INJECTION
# ==============================================================================
# Why it's being done differently:
# Your fine-tuned 2022 model weights (.pt files) contain specialized legacy
# custom tracking objects (like Metrics classes). When PyTorch unpickles these
# files during execution, it throws a ModuleNotFoundError because it can't find
# the source code modules. Injecting 'src' into Python's system path permanently
# bridges this map for all downstream execution steps.

import os
import sys
sys.path.append(os.path.abspath("src"))
print("Runtime paths synchronized successfully! Workspace ready for pipeline.")

Runtime paths synchronized successfully! Workspace ready for pipeline.


In [ ]:
# ==============================================================================
# CELL 5B: LOCAL TRANSFORMER MODEL RESOLUTION (ROBUST VERSION)
# ==============================================================================
# Why this exists:
# Ensures the Hugging Face backbone model is fully available locally and
# prevents silent failures during AutoConfig / AutoModel loading.

import os
from transformers import AutoConfig

# ----------------------------------------------------------------------
# 1. Resolve model directory safely (Colab + Drive safe)
# ----------------------------------------------------------------------
# Correctly resolve MODEL_DIR by using the known absolute path from Cell 2/4 context
# The files are downloaded to /content/drive/MyDrive/GitHub/inventory_2022/allenai/dsp_roberta_base_dapt_biomed_tapt_rct_500
REPO_ROOT = "/content/drive/MyDrive/GitHub/inventory_2022"
HF_MODEL_RELATIVE_PATH = "allenai/dsp_roberta_base_dapt_biomed_tapt_rct_500"
MODEL_DIR = os.path.join(REPO_ROOT, HF_MODEL_RELATIVE_PATH)


print("Resolved model path:", MODEL_DIR)

# ----------------------------------------------------------------------
# 2. Required HF files for a valid transformer model
# ----------------------------------------------------------------------
REQUIRED_FILES = [
    "config.json",
    "pytorch_model.bin",
    "vocab.json",
    "merges.txt",
    "tokenizer_config.json"
]

missing = [
    f for f in REQUIRED_FILES
    if not os.path.exists(os.path.join(MODEL_DIR, f))
]

if not os.path.exists(MODEL_DIR):
    raise FileNotFoundError(f"Model directory missing: {MODEL_DIR}")

if missing:
    raise FileNotFoundError(
        "❌ Incomplete Hugging Face model folder.\n"
        f"Missing files: {missing}\n"
        "Fix CELL 4 download step."
    )

print("✔ Local HF model folder complete")

# ----------------------------------------------------------------------
# 3. Load config safely (no internet, no HF lookup)
# ----------------------------------------------------------------------
config = AutoConfig.from_pretrained(
    MODEL_DIR,
    local_files_only=True,
    trust_remote_code=False
)

print("✔ Transformer config loaded successfully")
print("Model type:", getattr(config, "model_type", "unknown"))

Resolved model path: /content/drive/MyDrive/GitHub/inventory_2022/allenai/dsp_roberta_base_dapt_biomed_tapt_rct_500
✔ Local HF model folder complete
✔ Transformer config loaded successfully
Model type: roberta


# Setting up Configurations

Before running the automated pipelines, first update the configuration file `config/update_inventory.yml`. It can be accessed in Google Drive, though you may need to download it and edit it in a text editor such as Notepad, then reupload it.

* **Europe PMC query publication date range**: These are stored as variables `query_from_date` and `query_to_date` in that file. Note that the dates are inclusive. For example to get papers published in 2022, both of those variables should be 2022.
* **Previous inventory file**: During strict deduplication and flagging for manual review, the results of the previous inventory are taken into account. Specify the location of the most recent inventory output file in the variable `previous_inventory`.

# Running the pipeline
---
Now, we are ready to run the pipeline. It will take several minutes or even over an hour. Job progression will be shown in the output.

In [ ]:
%%bash
# ==============================================================================
# CELL 6: LAUNCH INVENTORY PIPELINE (70-90 MINS)
# ==============================================================================
# Why it's being done differently:
# We use the explicit force flag (-f) to clear out any stale, partial, or
# failed execution metadata, ensuring a comprehensive database recalculation.

cd /content/drive/MyDrive/GitHub/inventory_2022
make update_inventory


snakemake 	-s snakemake/update_inventory.smk 	--configfile config/update_inventory.yml -c1
Done. Saved predictions to out/new_query/classification/predictions.csv
Done. Saved predictions to out/new_query/ner/predictions.csv.
Done. Wrote output to out/new_query/url_extraction/predictions.csv.
Done processing names.
0 articles with no names removed.
Wrote output to out/new_query/processed_names/predictions.csv.
Done. Wrote output to out/new_query/initial_deduplication/predictions.csv.
Total number of flagged rows: 1313
Rows with duplicate names: 216
Rows with duplicate URLs: 90
Rows with low probability name: 1095
Wrote output to out/new_query/for_manual_review/predictions.csv.
Inventory flagged for manual review.
Once manual review is finished place file in out/new_query/manually_reviewed.


Building DAG of jobs...
Using shell: /usr/bin/bash
Provided cores: 1 (use --cores to define parallelism)
Rules claiming more threads will be scaled down.
Job stats:
job                      count    min threads    max threads
---------------------  -------  -------------  -------------
all                          1              1              1
classify_papers              1              1              1
extract_urls                 1              1              1
filter_positives             1              1              1
flag_for_review              1              1              1
initial_deduplication        1              1              1
ner_predict                  1              1              1
process_names                1              1              1
total                        8              1              1

Select jobs to execute...

[Fri Jul  3 23:36:33 2026]
rule classify_papers:
    input: out/classif_train_out/best/best_checkpt.txt, out/new_query/query_results.csv

# Selective Manual Review

After running the initial pipeline, the inventory has been flagged for selective manual review.

The file to be reviewed is located at:

`out/new_query/for_manual_review/predictions.csv`

Review the flagged columns according to the instruction sheet ([doi: 10.5281/zenodo.7768363](https://doi.org/10.5281/zenodo.7768363)), then place the manually reviewed file in the following folder:

`out/new_query/manually_reviewed/`

The file must still be named `predictions.csv`

# Processing Manual Review

Next, further processing is performed on the manually reviewed inventory.

In [ ]:
! make process_manually_reviewed_update

## Final inventory

The final inventory, including names, URLS, and metadata is found in the file:
*    `out/new_query/processed_countries/predictions.csv`